In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
%%time
import torch
import math
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling
from peft import PeftModel
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# --- Cấu hình ---
BASE_MODEL_ID = "Qwen/Qwen2-1.5B"
LORA_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/qwen_lora_adapter"
DATA_PATH = "/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl"
BATCH_SIZE = 16 # Tăng batch size để nhanh hơn
MAX_LENGTH = 256

# --- 1. Load Model & Tokenizer (Dùng float16 để nhanh) ---
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# --- 2. Chuẩn bị tập Test giữ nguyên thông tin source/domain ---
def get_test_dataset_with_metadata(file_path):
    dataset = load_dataset("json", data_files=file_path)["train"]
    unique_sources = list(set(dataset["source"]))
    test_lists = []
    for src in unique_sources:
        sub_ds = dataset.filter(lambda x: x["source"] == src)
        if len(sub_ds) < 3: continue
        split_1 = sub_ds.train_test_split(test_size=0.2, seed=42)
        test_lists.append(split_1["test"].train_test_split(test_size=0.5, seed=42)["test"])
    return concatenate_datasets(test_lists)

test_ds = get_test_dataset_with_metadata(DATA_PATH)

# --- 3. Hàm Evaluate chi tiết ---
def evaluate_detailed(model, tokenizer, dataset):
    # Khởi tạo lưu trữ kết quả: { 'group_name': { 'loss': 0, 'tokens': 0, 'hits': [0,0,0] } }
    stats = {"OVERALL": {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}}
    
    for row in tqdm(dataset, desc="Analyzing by Metadata"):
        source = row['source']
        domain = row['domain']
        groups = ["OVERALL", f"Source: {source}", f"Domain: {domain}"]
        
        # Tokenize mẫu đơn lẻ
        inputs = tokenizer(row["segmented_text"], truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(model.device)
        labels = inputs["input_ids"].clone()
        
        with torch.no_grad():
            outputs = model(**inputs, labels=labels)
            logits = outputs.logits
            
            # Shift để tính NWP
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            # Tính Hits
            _, top10_preds = shift_logits.topk(10, dim=-1)
            t1 = (top10_preds[..., 0] == shift_labels).sum().item()
            t5 = (top10_preds[..., :5] == shift_labels.unsqueeze(-1)).any(dim=-1).sum().item()
            t10 = (top10_preds == shift_labels.unsqueeze(-1)).any(dim=-1).sum().item()
            
            num_tokens = shift_labels.size(1)
            loss_val = outputs.loss.item() * num_tokens

            # Cập nhật vào các nhóm liên quan
            for g in groups:
                if g not in stats:
                    stats[g] = {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}
                stats[g]["loss"] += loss_val
                stats[g]["tokens"] += num_tokens
                stats[g]["top1"] += t1
                stats[g]["top5"] += t5
                stats[g]["top10"] += t10

    # Chuyển sang DataFrame để hiển thị
    report = []
    for name, s in stats.items():
        avg_loss = s["loss"] / s["tokens"]
        report.append({
            "Group": name,
            "Tokens": s["tokens"],
            "CCE": round(avg_loss, 4),
            "PPL": round(math.exp(avg_loss), 2),
            "Top-1 Acc (%)": round(s["top1"] / s["tokens"] * 100, 2),
            "Top-5 Acc (%)": round(s["top5"] / s["tokens"] * 100, 2),
            "Top-10 Acc (%)": round(s["top10"] / s["tokens"] * 100, 2),
        })
    return pd.DataFrame(report)

# Chạy và hiển thị
test_subset = test_ds.shuffle(seed=42).select(range(min(2000, len(test_ds))))

# Chạy lại hàm evaluate
df_results = evaluate_detailed(model, tokenizer, test_subset)
display(df_results.sort_values("Group"))

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

...

Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]

Analyzing by Metadata:   0%|          | 0/2000 [00:00<?, ?it/s]

,Group,Tokens,CCE,PPL,Top-1 Acc (%),Top-5 Acc (%),Top-10 Acc (%)
7,Domain: edu,28765,2.6919,14.76,46.61,67.32,75.05
2,Domain: ent,55396,3.5741,35.66,34.97,55.70,63.81
5,Domain: finance,42375,2.9362,18.84,43.48,64.50,71.99
8,Domain: tech,26885,2.7809,16.13,45.74,66.86,74.42
0,OVERALL,153421,3.0935,22.05,41.39,62.27,70.03
1,Source: forum_voz,30138,4.3724,79.23,26.33,45.09,53.02
6,Source: news_dantri,33327,2.8823,17.86,43.82,64.98,72.89
4,Source: news_thanhnien,58563,2.7312,15.35,45.57,67.18,74.86
3,Source: news_vnexpress,31393,2.7658,15.89,45.48,66.70,74.34


CPU times: user 3min 45s, sys: 27.7 s, total: 4min 13s
Wall time: 4min 12s


In [10]:
import random
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

print(" PHÂN TÍCH LỖI SAI")
print("="*80)

# ==========================================
# PHẦN 0: KHỞI TẠO LẠI DATALOADER (SỬA LỖI TẠI ĐÂY)
# ==========================================
# 1. Hàm tokenize
def tokenize_func(examples):
    return tokenizer(
        examples["segmented_text"], 
        truncation=True, 
        max_length=256
    )

# 2. Tokenize một tập nhỏ (khoảng 100 mẫu) để phân tích lỗi cho nhanh
# (Biến test_ds đã được tạo ở các cell trước đó)
sample_ds = test_ds.shuffle(seed=42).select(range(100))
sample_tokenized = sample_ds.map(tokenize_func, batched=True, remove_columns=sample_ds.column_names)

# 3. Tạo DataCollator và DataLoader
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
test_dataloader = DataLoader(sample_tokenized, batch_size=4, collate_fn=data_collator)


# ==========================================
# PHẦN 1: TÌM CÁC VỊ TRÍ MÔ HÌNH ĐOÁN SAI
# ==========================================
batch = next(iter(test_dataloader))
batch = {k: v.to(model.device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)
    logits = outputs.logits
    labels = batch["labels"]
    
    # Shift logits và labels (cơ chế Next Word Prediction)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Lấy dự đoán top 1 để so sánh với nhãn thật
    _, top1_preds = shift_logits.topk(1, dim=-1)
    
    # Cắt bớt input_ids tương ứng
    input_ids = batch["input_ids"][..., 1:] 

mismatches = []
for batch_idx in range(shift_labels.size(0)):
    for seq_idx in range(shift_labels.size(1)):
        true_label = shift_labels[batch_idx, seq_idx].item()
        if true_label == -100: # Bỏ qua các token padding
            continue
            
        pred_top1 = top1_preds[batch_idx, seq_idx, 0].item()
        if pred_top1 != true_label: # Nếu đoán sai Top-1
            mismatches.append((batch_idx, seq_idx, input_ids))

# ==========================================
# PHẦN 2: IN KẾT QUẢ ĐÃ ĐƯỢC "GỘP TỪ" DỄ ĐỌC
# ==========================================
if not mismatches:
    print("Không tìm thấy lỗi sai nào trong batch này!")
else:
    sample_errors = random.sample(mismatches, min(5, len(mismatches)))

    for b_idx, s_idx, inp_ids in sample_errors:
        # 1. Lấy ngữ cảnh (Context)
        start_idx = max(0, s_idx - 15) 
        context_tokens = inp_ids[b_idx, start_idx:s_idx]
        context_str = tokenizer.decode(context_tokens, skip_special_tokens=True).strip()
        
        # 2. Lấy Từ Thực Tế (Ground Truth)
        future_tokens = inp_ids[b_idx, s_idx:s_idx+10]
        future_str = tokenizer.decode(future_tokens, skip_special_tokens=True).strip()
        true_word = future_str.split(" ")[0].replace("_", " ") 
        
        # Bỏ qua nếu từ thực tế là dấu câu mồ côi
        if not true_word or true_word in [",", ".", "!", "?", ":", "-", '"']:
            continue
            
        # 3. Yêu cầu mô hình "viết nốt" bằng hàm generate
        input_tensor = context_tokens.unsqueeze(0).to(model.device)
        with torch.no_grad():
            gen_outputs = model.generate(
                input_tensor,
                max_new_tokens=6,       
                num_beams=5,            
                num_return_sequences=5, 
                pad_token_id=tokenizer.pad_token_id,
                early_stopping=True
            )
        
        # 4. Trích xuất các từ dự đoán duy nhất
        predicted_words = []
        for i in range(5):
            gen_new_tokens = gen_outputs[i, len(context_tokens):]
            gen_str = tokenizer.decode(gen_new_tokens, skip_special_tokens=True).strip()
            
            # Lấy trọn vẹn từ đầu tiên, loại bỏ dấu gạch dưới
            pred_word = gen_str.split(" ")[0].replace("_", " ")
            
            if pred_word and pred_word not in predicted_words:
                predicted_words.append(pred_word)
            if len(predicted_words) == 3: # Lấy đủ 3 gợi ý khác nhau thì dừng
                break
                
        clean_context = context_str.replace("_", " ")
        
        preds_str = " | ".join([f"{i+1}. '{w}'" for i, w in enumerate(predicted_words)])
        
        print(f" Ngữ cảnh: ...{clean_context} [???]")
        print(f" Thực tế : '{true_word}'")
        print(f" Đề xuất : {preds_str if preds_str else 'Không có đề xuất'}")
        print("-" * 80)

 PHÂN TÍCH LỖI SAI
 Ngữ cảnh: ...ách về cấu tạo thận hay bảng tuần hoàn nguyên t [???]
 Thực tế : 'ố'
 Đề xuất : 1. 'ùy'
--------------------------------------------------------------------------------
 Ngữ cảnh: ..., Phương Anh Đào - đôi " tình nhân màn  [???]
 Thực tế : 'ảnh'
 Đề xuất : 1. 'đèn'
--------------------------------------------------------------------------------
 Ngữ cảnh: ...Thiện Nhân , Trần Công Minh trong mắt loé h [???]
 Thực tế : 'àn'
 Đề xuất : 1. 'ào hùng'
--------------------------------------------------------------------------------
 Ngữ cảnh: ...dạy 3 món quen mà không dễ làm , đó là giò [???]
 Thực tế : 'thủ'
 Đề xuất : 1. 'chả'
--------------------------------------------------------------------------------
 Ngữ cảnh: ..., đánh dấu lần đầu hợp tác của nghệ sĩ Xuân H [???]
 Thực tế : 'inh'
 Đề xuất : 1. 'oan'
--------------------------------------------------------------------------------


In [13]:
%%time
import torch
import torch.nn as nn
import math
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from transformers import PreTrainedTokenizerFast
from tqdm.auto import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & HYPERPARAMETERS (Trích từ Config của bạn)
# ==========================================
TOKENIZER_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/bpe_vi_tokenizer"
MODEL_PATH = "/kaggle/input/datasets/truongminh3105/my-nlp-models/gru_nwp_v2_epoch_5.pt"
DATA_PATH = "/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl"

VOCAB_SIZE = 30000
EMBED_DIM = 1024
HIDDEN_DIM = 1024
NUM_LAYERS = 2
DROPOUT = 0.2
BLOCK_SIZE = 128 # Max length của GRU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. KHỞI TẠO LẠI KIẾN TRÚC GRU
# ==========================================
class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.fc.weight = self.embedding.weight # Tie weights

    def forward(self, input_ids, hidden=None):
        embedded = self.dropout(self.embedding(input_ids))
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(self.dropout(output))
        return logits, hidden

# ==========================================
# 3. LOAD TOKENIZER & MODEL WEIGHTS
# ==========================================
print("Đang load Tokenizer và mô hình GRU...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_PATH)
pad_idx = tokenizer.pad_token_id

# Khởi tạo khung mô hình
model = GRULanguageModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, pad_idx).to(device)

# Load weights từ file .pt
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)

# SỬA LỖI TẠI ĐÂY: Thêm logic kiểm tra cấu trúc file checkpoint
if 'model_state_dict' in checkpoint:
    # Nếu lưu dạng dictionary (theo code bạn cung cấp)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    # Nếu file .pt chỉ lưu trực tiếp state_dict (như bản v2 bạn đang dùng)
    model.load_state_dict(checkpoint)

model.eval()

# ==========================================
# 4. CHUẨN BỊ TẬP TEST (Kèm Metadata)
# ==========================================
def get_test_dataset_with_metadata(file_path):
    print("Đang tải và chia dữ liệu (Giữ nguyên cấu trúc Metadata)...")
    dataset = load_dataset("json", data_files=file_path)["train"]
    unique_sources = list(set(dataset["source"]))
    test_lists = []
    for src in unique_sources:
        sub_ds = dataset.filter(lambda x: x["source"] == src)
        if len(sub_ds) < 3: continue
        split_1 = sub_ds.train_test_split(test_size=0.2, seed=42)
        test_lists.append(split_1["test"].train_test_split(test_size=0.5, seed=42)["test"])
    return concatenate_datasets(test_lists)

test_ds = get_test_dataset_with_metadata(DATA_PATH)

# Lấy ngẫu nhiên 2000 mẫu để phân tích (Trộn đều để không bị kẹt ở 1 nguồn)
test_subset = test_ds.shuffle(seed=42).select(range(min(2000, len(test_ds))))

# ==========================================
# 5. HÀM ĐÁNH GIÁ (EVALUATE) CHO GRU
# ==========================================
def evaluate_gru_detailed(model, tokenizer, dataset, pad_idx):
    stats = {"OVERALL": {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}}
    loss_fct = nn.CrossEntropyLoss(ignore_index=pad_idx, reduction='sum') # Tính tổng loss để chia trung bình sau
    
    for row in tqdm(dataset, desc="Evaluating GRU"):
        source = row['source']
        domain = row['domain']
        groups = ["OVERALL", f"Source: {source}", f"Domain: {domain}"]
        
        # Tokenize
        inputs = tokenizer(row["segmented_text"], truncation=True, max_length=BLOCK_SIZE, return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)
        
        # Bỏ qua nếu câu quá ngắn (chỉ có 1 token) không đủ dự đoán
        if input_ids.size(1) < 2:
            continue
            
        with torch.no_grad():
            # Pass qua GRU
            logits, _ = model(input_ids)
            
            # Shift (Input -> Logits -> Labels)
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()
            
            # Chỉ tính toán các token không phải là padding
            mask = shift_labels != pad_idx
            valid_labels = shift_labels[mask]
            valid_logits = shift_logits[mask]
            
            num_tokens = mask.sum().item()
            if num_tokens == 0: continue
            
            # Tính tổng Cross Entropy Loss cho các token hợp lệ
            total_loss = loss_fct(shift_logits.view(-1, VOCAB_SIZE), shift_labels.view(-1)).item()
            
            # Tính Hits
            _, top10_preds = valid_logits.topk(10, dim=-1)
            t1 = (top10_preds[:, 0] == valid_labels).sum().item()
            t5 = (top10_preds[:, :5] == valid_labels.unsqueeze(1)).any(dim=1).sum().item()
            t10 = (top10_preds == valid_labels.unsqueeze(1)).any(dim=1).sum().item()

            # Cập nhật kết quả vào các nhóm tương ứng
            for g in groups:
                if g not in stats:
                    stats[g] = {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}
                stats[g]["loss"] += total_loss
                stats[g]["tokens"] += num_tokens
                stats[g]["top1"] += t1
                stats[g]["top5"] += t5
                stats[g]["top10"] += t10

    # Format báo cáo ra DataFrame
    report = []
    for name, s in stats.items():
        if s["tokens"] == 0: continue
        avg_loss = s["loss"] / s["tokens"]
        report.append({
            "Group": name,
            "Tokens": s["tokens"],
            "CCE": round(avg_loss, 4),
            "PPL": round(math.exp(min(avg_loss, 100)), 2), # Chặn overflow nếu PPL quá lớn
            "Top-1 Acc (%)": round(s["top1"] / s["tokens"] * 100, 2),
            "Top-5 Acc (%)": round(s["top5"] / s["tokens"] * 100, 2),
            "Top-10 Acc (%)": round(s["top10"] / s["tokens"] * 100, 2),
        })
    return pd.DataFrame(report)

# ==========================================
# 6. THỰC THI & HIỂN THỊ
# ==========================================
df_results = evaluate_gru_detailed(model, tokenizer, test_subset, pad_idx)
display(df_results.sort_values("Group"))

Đang load Tokenizer và mô hình GRU...
Đang tải và chia dữ liệu (Giữ nguyên cấu trúc Metadata)...


Evaluating GRU:   0%|          | 0/2000 [00:00<?, ?it/s]

,Group,Tokens,CCE,PPL,Top-1 Acc (%),Top-5 Acc (%),Top-10 Acc (%)
7,Domain: edu,23949,9.2147,10043.38,10.17,17.57,18.66
2,Domain: ent,46535,9.5932,14664.08,7.85,13.67,14.66
5,Domain: finance,33036,9.2869,10795.89,9.48,16.27,17.48
8,Domain: tech,22655,9.2506,10410.94,9.80,17.14,18.27
0,OVERALL,126175,9.3796,11844.65,9.07,15.71,16.80
1,Source: forum_voz,25745,9.8204,18405.49,6.03,10.70,11.71
6,Source: news_dantri,28644,9.3167,11121.97,9.76,16.90,18.04
4,Source: news_thanhnien,45854,9.2174,10071.03,9.99,17.25,18.29
3,Source: news_vnexpress,25932,9.2984,10920.59,9.68,16.66,17.87


CPU times: user 32.3 s, sys: 339 ms, total: 32.6 s
Wall time: 32.9 s


In [16]:
import random
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

print(" PHÂN TÍCH LỖI SAI CHO MÔ HÌNH GRU (CẤP ĐỘ TỪ)")
print("="*80)

# ==========================================
# PHẦN 0: KHỞI TẠO DATALOADER CHO GRU
# ==========================================
# 1. Lấy 100 mẫu ngẫu nhiên (Biến test_ds đã có ở cell trước)
sample_ds = test_ds.shuffle(seed=42).select(range(100))

# 2. Định nghĩa hàm Collate để tự động Padding cho batch
def gru_collate_fn(batch):
    # Lấy list các câu và tokenize
    input_ids_list = [torch.tensor(tokenizer.encode(item["segmented_text"])) for item in batch]
    # Pad các câu cho bằng nhau theo câu dài nhất trong batch
    padded_input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=pad_idx)
    return padded_input_ids

test_dataloader = DataLoader(sample_ds, batch_size=4, collate_fn=gru_collate_fn)

# ==========================================
# PHẦN 1: TÌM CÁC VỊ TRÍ MÔ HÌNH ĐOÁN SAI
# ==========================================
full_input_ids = next(iter(test_dataloader)).to(device)

with torch.no_grad():
    # Pass qua mô hình GRU
    logits, _ = model(full_input_ids)
    
    # Shift logits và labels
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_input_ids[..., 1:].contiguous()
    
    # Lấy dự đoán top 1
    _, top1_preds = shift_logits.topk(1, dim=-1)

mismatches = []
for batch_idx in range(shift_labels.size(0)):
    for seq_idx in range(shift_labels.size(1)):
        true_label = shift_labels[batch_idx, seq_idx].item()
        
        if true_label == pad_idx: # Bỏ qua padding
            continue
            
        pred_top1 = top1_preds[batch_idx, seq_idx, 0].item()
        if pred_top1 != true_label:
            # Lưu lại index của batch và vị trí seq_idx bị sai
            mismatches.append((batch_idx, seq_idx))

# ==========================================
# PHẦN 2: IN KẾT QUẢ & SINH GỢI Ý (GREEDY DECODING)
# ==========================================
if not mismatches:
    print("Không tìm thấy lỗi sai nào trong batch này!")
else:
    sample_errors = random.sample(mismatches, min(5, len(mismatches)))

    for b_idx, s_idx in sample_errors:
        # 1. Lấy ngữ cảnh (Context bao gồm từ đầu cho tới token hiện tại)
        start_idx = max(0, s_idx - 15) 
        context_tokens = full_input_ids[b_idx, start_idx : s_idx + 1]
        context_str = tokenizer.decode(context_tokens, skip_special_tokens=True).strip()
        
        # 2. Lấy Từ Thực Tế (Ground Truth là token nằm ở vị trí s_idx + 1)
        future_tokens = full_input_ids[b_idx, s_idx + 1 : s_idx + 5]
        future_str = tokenizer.decode(future_tokens, skip_special_tokens=True).strip()
        true_word = future_str.split(" ")[0].replace("_", " ") 
        
        # Bỏ qua nếu là dấu câu
        if not true_word or true_word in [",", ".", "!", "?", ":", "-", '"']:
            continue
            
        # 3. Yêu cầu GRU đề xuất 3 từ tiếp theo
        input_tensor = context_tokens.unsqueeze(0).to(device)
        predicted_words = []
        
        with torch.no_grad():
            # Lấy logits tại bước cuối cùng của context
            out_logits, hidden = model(input_tensor)
            next_token_logits = out_logits[0, -1, :]
            
            # Lấy Top 6 token có xác suất cao nhất phòng trường hợp trùng từ
            _, top_k_indices = torch.topk(next_token_logits, 6)
            
            for i in range(top_k_indices.size(0)):
                token_id = top_k_indices[i].item()
                
                # Bắt đầu greedy decode thêm vài token để hoàn thành 1 từ (nếu BPE băm quá nhỏ)
                seq = [token_id]
                curr_hidden = hidden
                curr_input = torch.tensor([[token_id]], device=device)
                
                for _ in range(3): # Sinh tối đa 3 sub-word nữa
                    out, curr_hidden = model(curr_input, curr_hidden)
                    next_tok = out[0, -1, :].argmax().item()
                    if next_tok == tokenizer.eos_token_id or next_tok == pad_idx:
                        break
                    seq.append(next_tok)
                    curr_input = torch.tensor([[next_tok]], device=device)
                
                # Giải mã chuỗi token thành chữ
                gen_str = tokenizer.decode(seq, skip_special_tokens=True).strip()
                pred_word = gen_str.split(" ")[0].replace("_", " ")
                
                if pred_word and pred_word not in predicted_words:
                    predicted_words.append(pred_word)
                if len(predicted_words) == 3: # Đủ 3 đề xuất thì dừng
                    break
                
        clean_context = context_str.replace("_", " ")
        preds_str = " | ".join([f"{i+1}. '{w}'" for i, w in enumerate(predicted_words)])
        
        print(f" Ngữ cảnh: ...{clean_context} [???]")
        print(f" Thực tế : '{true_word}'")
        print(f" Đề xuất : {preds_str if preds_str else 'Không có đề xuất'}")
        print("-" * 80)

 PHÂN TÍCH LỖI SAI CHO MÔ HÌNH GRU (CẤP ĐỘ TỪ)
 Ngữ cảnh: ... Nguyên đán có sự tham gia của hai dự án . Mùi phở [???]
 Thực tế : 'là'
 Đề xuất : 1. ' Khh�ắm' | 2. 'à' | 3. 'h�ắm'
--------------------------------------------------------------------------------
 Ngữ cảnh: ... sĩ Xuân Hinh và diễn viên Thu Trang . Báu vật [???]
 Thực tế : 'trời'
 Đề xuất : 1. 'h�(phố ' | 2. 'à' | 3. 'ên'
--------------------------------------------------------------------------------
 Ngữ cảnh: ...Sách về cấu  [???]
 Thực tế : 'tạo'
 Đề xuất : 1. 'Sà' | 2. 'Pà' | 3. 'khúcà'
--------------------------------------------------------------------------------
 Ngữ cảnh: ...hôm nay dạy 3 món quen mà không dễ làm , đó là giò thủ [???]
 Thực tế : 'gói'
 Đề xuất : 1. ' ' | 2. 'àquyánh' | 3. 'h�"bố'
--------------------------------------------------------------------------------


In [19]:
import os
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.cuda.amp import autocast
import pandas as pd
from datasets import load_dataset, concatenate_datasets
import random

print(" KHỞI TẠO ĐÁNH GIÁ MÔ HÌNH TCN")
print("="*80)

# ==========================================
# 1. CẤU HÌNH & HYPERPARAMETERS
# ==========================================
MODEL_PATH = '/kaggle/input/datasets/truongminh3105/my-nlp-models/tcn_vn_model_final.pth'
VOCAB_PATH = '/kaggle/input/datasets/truongminh3105/my-nlp-models/vocab_train.json'
DATA_PATH = '/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl'

EMBED_SIZE = 256
NUM_CHANNELS = [128, 256, 512, 256] 
KERNEL_SIZE = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. KHÔI PHỤC VOCABULARY TỪ FILE JSON
# ==========================================
class Vocabulary:
    def __init__(self):
        self.pad_token, self.pad_idx = '<pad>', 0
        self.unk_token, self.unk_idx = '<unk>', 1
        self.sos_token, self.sos_idx = '<sos>', 2
        self.eos_token, self.eos_idx = '<eos>', 3
        self.word2idx = {}
        self.idx2word = {}

    @classmethod
    def load_from_json(cls, path):
        vocab = cls()
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Tùy thuộc vào định dạng bạn lưu (chỉ list từ hay dict map)
        if isinstance(data, dict) and "word2idx" in data:
            vocab.word2idx = data["word2idx"]
        elif isinstance(data, dict):
            vocab.word2idx = data
        
        # Khôi phục idx2word đảm bảo key là số nguyên
        vocab.idx2word = {int(idx): word for word, idx in vocab.word2idx.items()}
        return vocab

    def encode(self, text, add_sos_eos=True):
        tokens = text.split()
        seq = [self.word2idx.get(w, self.unk_idx) for w in tokens]
        if add_sos_eos:
            seq = [self.sos_idx] + seq + [self.eos_idx]
        return seq

    def decode(self, indices):
        return " ".join([self.idx2word.get(int(idx), self.unk_token) for idx in indices])

vocab = Vocabulary.load_from_json(VOCAB_PATH)
VOCAB_SIZE = len(vocab.word2idx)
print(f"Đã nạp Vocabulary với {VOCAB_SIZE} từ.")

# ==========================================
# 3. KHÔI PHỤC KIẾN TRÚC TCN MODEL
# ==========================================
model = TCNLanguageModel(vocab_size=VOCAB_SIZE, embed_size=EMBED_SIZE, num_channels=NUM_CHANNELS, kernel_size=KERNEL_SIZE)

# Tải state_dict từ file
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)

# Sửa lỗi tiền tố "module." nếu train bằng DataParallel
if list(state_dict.keys())[0].startswith("module."):
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

# FIX LỖI PYTORCH WEIGHT_NORM VERSION MISMATCH (ĐÃ ĐẢO CHIỀU CHÍNH XÁC)
fixed_state_dict = {}
for k, v in state_dict.items():
    # Ánh xạ lại chính xác:
    # original0 -> weight_g (Tensor nhỏ: Độ lớn)
    # original1 -> weight_v (Tensor lớn: Hướng)
    if 'parametrizations.weight.original0' in k:
        new_k = k.replace('parametrizations.weight.original0', 'weight_g')
    elif 'parametrizations.weight.original1' in k:
        new_k = k.replace('parametrizations.weight.original1', 'weight_v')
    else:
        new_k = k
    fixed_state_dict[new_k] = v

# Load lại state_dict đã được sửa
model.load_state_dict(fixed_state_dict)
model.to(device)
model.eval()
print(" Đã load trọng số mô hình TCN thành công!")

# ==========================================
# 4. CHUẨN BỊ TẬP TEST KÈM METADATA
# ==========================================
print("Đang nạp tập Data (giữ nguyên Metadata)...")
dataset = load_dataset("json", data_files=DATA_PATH)["train"]
unique_sources = list(set(dataset["source"]))
test_lists = []
for src in unique_sources:
    sub_ds = dataset.filter(lambda x: x["source"] == src)
    if len(sub_ds) < 3: continue
    split_1 = sub_ds.train_test_split(test_size=0.2, seed=42)
    test_lists.append(split_1["test"].train_test_split(test_size=0.5, seed=42)["test"])

test_ds = concatenate_datasets(test_lists)
test_subset = test_ds.shuffle(seed=42).select(range(min(2000, len(test_ds))))


# ==========================================
# 5. ĐÁNH GIÁ CHI TIẾT SOURCE & DOMAIN
# ==========================================
def evaluate_tcn_detailed(model, vocab, dataset):
    stats = {"OVERALL": {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}}
    loss_fct = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx, reduction='sum')
    
    # Do TCN nhận độ dài tùy ý, ta có thể đánh giá từng dòng thay vì padding batch để kết quả chính xác tuyệt đối
    for row in dataset:
        source = row['source']
        domain = row.get('domain', 'unknown')
        groups = ["OVERALL", f"Source: {source}", f"Domain: {domain}"]
        
        # Tokenize (Có SOS và EOS)
        encoded = vocab.encode(row["segmented_text"], add_sos_eos=True)
        if len(encoded) < 3: continue # Quá ngắn
        
        # Với Casual TCN: input là [:-1], label là [1:]
        x = torch.tensor([encoded[:-1]], dtype=torch.long).to(device)
        y = torch.tensor([encoded[1:]], dtype=torch.long).to(device)
        
        with torch.no_grad():
            with autocast():
                logits = model(x)
            
            # Tính loss
            loss_val = loss_fct(logits.view(-1, VOCAB_SIZE), y.view(-1)).item()
            num_tokens = y.size(1)
            
            # Tính hits (Accuracy)
            _, top10_preds = logits.topk(10, dim=-1)
            t1 = (top10_preds[:, :, 0] == y).sum().item()
            t5 = (top10_preds[:, :, :5] == y.unsqueeze(-1)).any(dim=-1).sum().item()
            t10 = (top10_preds == y.unsqueeze(-1)).any(dim=-1).sum().item()
            
            for g in groups:
                if g not in stats:
                    stats[g] = {"loss": 0, "tokens": 0, "top1": 0, "top5": 0, "top10": 0}
                stats[g]["loss"] += loss_val
                stats[g]["tokens"] += num_tokens
                stats[g]["top1"] += t1
                stats[g]["top5"] += t5
                stats[g]["top10"] += t10

    report = []
    for name, s in stats.items():
        if s["tokens"] == 0: continue
        avg_loss = s["loss"] / s["tokens"]
        report.append({
            "Group": name,
            "Tokens": s["tokens"],
            "CCE": round(avg_loss, 4),
            "PPL": round(math.exp(min(avg_loss, 100)), 2),
            "Top-1 Acc (%)": round(s["top1"] / s["tokens"] * 100, 2),
            "Top-5 Acc (%)": round(s["top5"] / s["tokens"] * 100, 2),
            "Top-10 Acc (%)": round(s["top10"] / s["tokens"] * 100, 2),
        })
    return pd.DataFrame(report)

print("\nĐang tính toán các chỉ số Evaluation...")
df_results = evaluate_tcn_detailed(model, vocab, test_subset)
display(df_results.sort_values("Group"))




 KHỞI TẠO ĐÁNH GIÁ MÔ HÌNH TCN
Đã nạp Vocabulary với 30000 từ.
🎉 Đã load trọng số mô hình TCN thành công!
Đang nạp tập Data (giữ nguyên Metadata)...


Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/797461 [00:00<?, ? examples/s]


Đang tính toán các chỉ số Evaluation...


/tmp/ipykernel_57/3361662867.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


,Group,Tokens,CCE,PPL,Top-1 Acc (%),Top-5 Acc (%),Top-10 Acc (%)
7,Domain: edu,15589,4.0776,59.01,25.63,47.51,56.54
2,Domain: ent,33958,4.9451,140.48,19.20,37.71,46.38
5,Domain: finance,23949,4.3528,77.69,23.60,44.04,52.61
8,Domain: tech,14848,4.2456,69.80,24.28,45.23,54.22
0,OVERALL,88344,4.5139,91.28,22.38,42.42,51.18
1,Source: forum_voz,20477,5.7459,312.90,12.14,27.79,35.97
6,Source: news_dantri,18538,4.2344,69.02,24.94,46.26,55.33
4,Source: news_thanhnien,32039,4.0940,59.98,25.77,47.21,56.08
3,Source: news_vnexpress,17290,4.1325,62.34,25.48,46.74,55.66



🔍 PHÂN TÍCH LỖI SAI (TOP-3 GỢI Ý CHO MÔ HÌNH TCN)
📝 Ngữ cảnh: ...Đây là chiến thắng thứ 13 liên tiếp tại Saudi Pro League của Al Nassr . [???]
✅ Thực tế : 'Ronaldo'
💡 Đề xuất : 1. 'Trong' | 2. 'Tuy nhiên' | 3. 'Đây'
--------------------------------------------------------------------------------
📝 Ngữ cảnh: ...và đồng đội củng cố vị trí dẫn đầu với 70 điểm sau 27 trận , hơn Al Hilal sáu điểm [???]
✅ Thực tế : 'và'
💡 Đề xuất : 1. '.' | 2. ',' | 3. 'sau'
--------------------------------------------------------------------------------
📝 Ngữ cảnh: ...củng cố vị trí dẫn đầu với 70 điểm sau 27 trận , hơn Al Hilal sáu điểm và Al Ahli [???]
✅ Thực tế : 'tám'
💡 Đề xuất : 1. 'đang' | 2. 'đã' | 3. 'vẫn'
--------------------------------------------------------------------------------
📝 Ngữ cảnh: ...Đây là chiến thắng thứ 13 liên tiếp tại Saudi Pro League của Al Nassr . Ronaldo và [???]
✅ Thực tế : 'đồng đội'
💡 Đề xuất : 1. '<unk>' | 2. 'Sinner' | 3. 'Francisco'
-----------------------------------

/tmp/ipykernel_57/3361662867.py:202: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_57/3361662867.py:233: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
# ==========================================
# 6. PHÂN TÍCH LỖI SAI CẤP ĐỘ TỪ (TCN)
# ==========================================
print("\n" + "="*80)
print(" PHÂN TÍCH LỖI SAI (TOP-3 GỢI Ý CHO MÔ HÌNH TCN)")
print("="*80)

# Trích xuất 1 mẫu ngẫu nhiên dài dài chút từ subset
sample = None
for row in test_subset.shuffle():
    if len(row['segmented_text'].split()) > 30:
        sample = row['segmented_text']
        break

encoded_sample = vocab.encode(sample, add_sos_eos=True)
x_sample = torch.tensor([encoded_sample[:-1]], dtype=torch.long).to(device)
y_sample = torch.tensor([encoded_sample[1:]], dtype=torch.long).to(device)

with torch.no_grad():
    with autocast():
        logits = model(x_sample)
    _, top1_preds = logits.topk(1, dim=-1)

mismatches = []
for seq_idx in range(y_sample.size(1)):
    true_label = y_sample[0, seq_idx].item()
    pred_top1 = top1_preds[0, seq_idx, 0].item()
    
    if true_label not in [vocab.pad_idx, vocab.unk_idx, vocab.eos_idx] and pred_top1 != true_label:
        mismatches.append(seq_idx)

if not mismatches:
    print("Mô hình không đoán sai từ nào trong mẫu thử nghiệm này!")
else:
    sample_errors = random.sample(mismatches, min(5, len(mismatches)))
    
    for s_idx in sample_errors:
        # Ngữ cảnh: Lấy khoảng 15 token trước điểm sai
        start_idx = max(0, s_idx - 15)
        context_tokens = encoded_sample[:s_idx + 1] # Gồm luôn cả token <sos>
        context_str = vocab.decode(encoded_sample[start_idx:s_idx + 1]).replace("<sos>", "").strip()
        
        # Từ đúng thực tế
        true_word = vocab.idx2word.get(encoded_sample[s_idx + 1], "")
        if true_word in [",", ".", "!", "?", ":", "-"]: continue
            
        # Lấy Top 3 dự đoán của mô hình TCN tại vị trí này
        # Truyền context thực tế vào để lấy logit bước cuối cùng
        input_tensor = torch.tensor([context_tokens], dtype=torch.long).to(device)
        with torch.no_grad():
            with autocast():
                out_logits = model(input_tensor)
            
            # Lấy vị trí thời gian cuối cùng
            next_token_logits = out_logits[0, -1, :]
            
            # Lấy xác suất
            probs = F.softmax(next_token_logits, dim=-1)
            top_k_probs, top_k_indices = torch.topk(probs, 5) # Lấy top 5 dự phòng trùng lặp
            
            predicted_words = []
            for idx in top_k_indices:
                word = vocab.idx2word.get(idx.item(), "")
                if word not in [vocab.pad_idx, vocab.unk_idx, vocab.eos_idx, vocab.sos_idx]:
                    clean_word = word.replace("_", " ")
                    if clean_word and clean_word not in predicted_words:
                        predicted_words.append(clean_word)
                if len(predicted_words) == 3: break
                
        print(f" Ngữ cảnh: ...{context_str.replace('_', ' ')} [???]")
        print(f" Thực tế : '{true_word.replace('_', ' ')}'")
        preds_str = " | ".join([f"{i+1}. '{w}'" for i, w in enumerate(predicted_words)])
        print(f" Đề xuất : {preds_str}")
        print("-" * 80)